<a href="https://colab.research.google.com/github/cy6erlizard/landsat-lake-clarity/blob/main/notebooks/01_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1 - Audit the published data

Fetches the LAGOS-US LANDSAT matchup table and LAGOS-US LOCUS lake metadata from EDI
onto Drive, converts them to Parquet once, and produces figures F1 to F5.

**Run this in Colab, not locally.** The EDI pull is ~550 MB and Colab sits in a Google
datacenter; a home connection takes hours for the same bytes.

Expectations were written down in `PLAN.md` before any of this ran. Two were already
wrong. That is the point of writing them down.

In [1]:
# Cell 1 of 4: code
import os, pathlib, subprocess, sys

REPO = "https://github.com/cy6erlizard/landsat-lake-clarity.git"
ROOT = pathlib.Path("/content/landsat-lake-clarity")

if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / "src"))

In [2]:
# Cell 2 of 4: Drive, and point the package's data root at it
from google.colab import drive
drive.mount("/content/drive")

os.environ["LAKECLARITY_DATA"] = "/content/drive/MyDrive/lake-clarity"
pathlib.Path(os.environ["LAKECLARITY_DATA"]).mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [3]:

# Cell 3 of 4: fetch once, convert once. Both steps skip if already on Drive.
import pandas as pd
from lakeclarity import audit, config, edi, features, locus, viz

viz.use_style()

edi.download("matchups")
edi.download("lake_information")
edi.download("lake_characteristics")

matchups = edi.load_matchups()
lakes = locus.load_lakes()

features.check_schema(matchups)
print(f"{len(matchups):,} matchups x {matchups.shape[1]} columns")
print(f"{len(lakes):,} lakes in LOCUS")
assert matchups.shape[1] == config.N_COLUMNS_PUBLISHED

matchups:   0%|          | 0.00/426M [00:00<?, ?B/s]

lake_information:   0%|          | 0.00/128M [00:00<?, ?B/s]

lake_characteristics:   0%|          | 0.00/100M [00:00<?, ?B/s]

matchups.csv -> parquet: 0chunk [00:00, ?chunk/s]

723,206 matchups x 54 columns
479,950 lakes in LOCUS


In [4]:
# Cell 4 of 4: the audit itself
avail = audit.secchi_availability(matchups)
print(avail)

comp = audit.completeness(matchups)
comp.to_csv(config.TABLE_DIR / "T01_completeness.csv")
print("\nconstant columns:", comp.index[comp.is_constant].tolist())

neg = audit.negative_reflectance_report(matchups)
neg.to_csv(config.TABLE_DIR / "T02_negative_reflectance.csv", index=False)
print("\n", neg.to_string(index=False))
print("\nany band negative: %.3f%% of Secchi matchups" % neg.attrs["any_band_negative_pct"])
print("mean Secchi where a band is negative : %.3f m" % neg.attrs["mean_secchi_any_negative"])
print("mean Secchi where all bands positive : %.3f m" % neg.attrs["mean_secchi_no_negative"])
print("\nIf the first is LARGER than the second, the standard quality filter is deleting")
print("the clear end of the distribution, and the whole lineage trained on a biased sample.")

for fn, fid, slug in [
    (audit.fig_missingness, "F01", "missingness"),
    (audit.fig_day_diff, "F02", "day_diff_window"),
    (audit.fig_coverage_by_satellite, "F03", "coverage_by_satellite"),
    (audit.fig_pixelcount, "F04", "pixelcount"),
    (audit.fig_secchi_transform, "F05", "secchi_transform"),
]:
    print("wrote", viz.save(fn(matchups), fid, slug).name)

{'n_matchups': 723206, 'n_with_secchi': 666060, 'pct_with_secchi': np.float64(92.1), 'n_lakes_with_secchi': 12735}

constant columns: ['IMAGE_QUALITY_TIRS']

  band  n_negative  pct_negative  mean_secchi_when_negative  mean_secchi_when_positive
 Blue        1225         0.184                      2.756                      2.733
Green         187         0.028                      2.468                      2.734
  Red         917         0.138                      3.446                      2.733
  NIR         476         0.071                      3.105                      2.733
SWIR1         105         0.016                      3.264                      2.733
SWIR2          12         0.002                      2.294                      2.734

any band negative: 0.258% of Secchi matchups
mean Secchi where a band is negative : 3.053 m
mean Secchi where all bands positive : 2.733 m

If the first is LARGER than the second, the standard quality filter is deleting
the clear end of t